# UniLumos BSS Official All-Demos Full Run

Colab-first notebook for the formal UniLumos official-demo BSS experiment. It uses only official `examples/examples_refined.csv` rows, runs the selected `abc` mode, writes outputs to Google Drive, computes reference-normalized RGB-L1 closure metrics, and generates support-aware paper tables.

## 1. Configure Sources, Drive Paths, And Run Flags

This notebook is intentionally resumable. Keep `RUN_FULL = True` when you are ready to run the full 20-scenario ladder. Set `FULL_LIMIT` to a small integer if you want to test only a few pending rows.

In [ ]:
from pathlib import Path
import os
import sys
import json
import csv
import math
import subprocess
import shutil
from collections import Counter, defaultdict

GITHUB_REPO = "https://github.com/WANG-Ruipeng/Lumos-Custom.git"
BRANCH = "bss-unilumos-official-smoke"
LOCAL_REPO_ROOT = Path("/content/Lumos-Custom")

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_WEIGHTS_ROOT = DRIVE_ROOT / "Colab_Projects" / "UniLumos" / "weights"
LOCAL_WEIGHTS_ROOT = LOCAL_REPO_ROOT / "UniLumos" / "UniLumos" / "weights"
WEIGHTS_ROOT = LOCAL_WEIGHTS_ROOT
EXP_ROOT = DRIVE_ROOT / "Colab_Projects" / "UniLumos-BSS-Runs" / "model_b_unilumos_official_all_demos_bss_v1"

HF_REPO_ID = "Alibaba-DAMO-Academy/UniLumos"
DOWNLOAD_WEIGHTS_TO_DRIVE = False
STAGE_WEIGHTS_TO_LOCAL = True
RUN_INSTALL = False
INSTALL_TORCH_MODE = "auto"  # auto, requirements, or cu128 for RTX PRO 6000 Blackwell

RUN_AUDIT = True
RUN_MAKE_MANIFEST = True
RESET_MANIFEST = False  # keep False for resume; True recreates manifests from scratch
RUN_SMOKE_FIRST = False  # direct full run; set True only if you want a smoke gate
RUN_FULL = True
FULL_LIMIT = None  # set to an int for a short resume test, e.g. 4
ONLY_METHODS = ""  # e.g. "bss10,bss12"; empty means all methods
RUN_METRICS = True
RUN_TABLES = True
RUN_FINAL_SUMMARY = True

print("GITHUB_REPO =", GITHUB_REPO)
print("BRANCH =", BRANCH)
print("LOCAL_REPO_ROOT =", LOCAL_REPO_ROOT)
print("DRIVE_WEIGHTS_ROOT =", DRIVE_WEIGHTS_ROOT)
print("LOCAL_WEIGHTS_ROOT =", LOCAL_WEIGHTS_ROOT)
print("WEIGHTS_ROOT =", WEIGHTS_ROOT)
print("EXP_ROOT =", EXP_ROOT)
print("INSTALL_TORCH_MODE =", INSTALL_TORCH_MODE)
print("RUN_FULL =", RUN_FULL, "FULL_LIMIT =", FULL_LIMIT, "ONLY_METHODS =", ONLY_METHODS)

## 2. Mount Google Drive

Weights and experiment outputs live on Drive. Code is cloned into Colab local `/content`.

In [ ]:
IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not running in Colab; Drive mount skipped.")

print("Drive available:", DRIVE_ROOT.exists())

## 3. Clone Or Pull The Experiment Branch

Rerun this cell whenever new code is pushed to the fork.

In [ ]:
def run(cmd, cwd=None, check=True):
    print("$", " ".join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if LOCAL_REPO_ROOT.exists() and (LOCAL_REPO_ROOT / ".git").exists():
    run(["git", "fetch", "origin", BRANCH], cwd=LOCAL_REPO_ROOT)
    run(["git", "checkout", BRANCH], cwd=LOCAL_REPO_ROOT)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=LOCAL_REPO_ROOT)
else:
    LOCAL_REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "-b", BRANCH, GITHUB_REPO, str(LOCAL_REPO_ROOT)])

TASK_ROOT = LOCAL_REPO_ROOT / "UniLumos"
CODE_ROOT = TASK_ROOT / "UniLumos"
assert (CODE_ROOT / "unilumos_infer_abc.py").exists(), CODE_ROOT

os.environ["PYTHONPATH"] = f"{TASK_ROOT}:{CODE_ROOT}:" + os.environ.get("PYTHONPATH", "")
if str(TASK_ROOT) not in sys.path:
    sys.path.insert(0, str(TASK_ROOT))
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=LOCAL_REPO_ROOT, text=True).strip()
print("TASK_ROOT =", TASK_ROOT)
print("CODE_ROOT =", CODE_ROOT)
print("Checked out commit =", commit)

## 4. Optional Dependency Install

On RTX PRO 6000 / Blackwell, `INSTALL_TORCH_MODE = "auto"` installs PyTorch CUDA 12.8 wheels. On H100/A100 it keeps the official requirements path.

In [ ]:
if RUN_INSTALL:
    install_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "install_colab_unilumos_deps.py"),
        "--torch-mode", INSTALL_TORCH_MODE,
    ]
    subprocess.check_call(install_cmd)
else:
    print("Dependency install skipped. Set RUN_INSTALL = True if needed.")

## 5. Optional One-Time Weight Download To Drive

Use only if Drive does not already contain UniLumos weights. The Hugging Face repo is gated, so accept model terms first and log in when prompted.

In [ ]:
if DOWNLOAD_WEIGHTS_TO_DRIVE:
    DRIVE_WEIGHTS_ROOT.mkdir(parents=True, exist_ok=True)
    run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import notebook_login, snapshot_download

    notebook_login()
    snapshot_download(
        repo_id=HF_REPO_ID,
        local_dir=str(DRIVE_WEIGHTS_ROOT),
        local_dir_use_symlinks=False,
    )
    print("Downloaded weights to Drive:", DRIVE_WEIGHTS_ROOT)
else:
    print("DOWNLOAD_WEIGHTS_TO_DRIVE is False; one-time HF download skipped.")

## 6. Stage Weights To Local Runtime

Every fresh Colab runtime should stage weights from Drive to local disk before model execution.

In [ ]:
def sync_tree(src: Path, dst: Path) -> None:
    if not src.exists():
        raise FileNotFoundError(f"Source does not exist: {src}")
    dst.mkdir(parents=True, exist_ok=True)
    if shutil.which("rsync"):
        run(["rsync", "-a", "--info=progress2", str(src) + "/", str(dst) + "/"])
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)

if STAGE_WEIGHTS_TO_LOCAL:
    sync_tree(DRIVE_WEIGHTS_ROOT, LOCAL_WEIGHTS_ROOT)
    WEIGHTS_ROOT = LOCAL_WEIGHTS_ROOT
else:
    WEIGHTS_ROOT = DRIVE_WEIGHTS_ROOT

print("Using WEIGHTS_ROOT:", WEIGHTS_ROOT)

## 7. Verify Code, Official Examples, And Weights

Do not run smoke/full until every required code and weight path is `OK`.

In [ ]:
required_code = [
    TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "make_manifest_unilumos_official_all_demos.py",
    TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "run_manifest.py",
    TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "compute_metrics_against_ref.py",
    TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "make_cross_model_tables.py",
    CODE_ROOT / "examples" / "examples_refined.csv",
    CODE_ROOT / "unilumos_infer_abc.py",
    CODE_ROOT / "src" / "schedulers" / "RFLOW_WANX21_T2V.py",
]
required_weights = [
    WEIGHTS_ROOT / "models_t5_umt5-xxl-enc-bf16.pth",
    WEIGHTS_ROOT / "umt5-xxl",
    WEIGHTS_ROOT / "vae.pth",
    WEIGHTS_ROOT / "unilumos.pt",
]

missing = []
for path in required_code + required_weights:
    ok = path.exists()
    print(("OK      " if ok else "MISSING "), path)
    if not ok:
        missing.append(str(path))

if missing:
    raise FileNotFoundError("Missing required paths; see printed MISSING entries.")

## 8. Audit Official Demo Scenarios

Confirms that selected mode is `abc` and official scenarios come directly from `examples/examples_refined.csv`.

In [ ]:
if RUN_AUDIT:
    import csv

    examples_csv = CODE_ROOT / "examples" / "examples_refined.csv"
    with examples_csv.open("r", encoding="utf-8", newline="") as handle:
        official_rows = list(csv.DictReader(handle))
    report_dir = EXP_ROOT / "reports"
    report_dir.mkdir(parents=True, exist_ok=True)
    audit_path = report_dir / "00_official_demo_audit.md"
    lines = [
        "# UniLumos Official Demo Audit",
        "",
        f"- selected mode: `abc`",
        f"- official source: `{examples_csv}`",
        f"- official scenarios: {len(official_rows)}",
        f"- checked out commit: `{commit}`",
        f"- experiment root: `{EXP_ROOT}`",
        "",
        "| scenario_id | foreground | background |",
        "|---|---|---|",
    ]
    for row in official_rows:
        scenario_id = f"{row.get('example_folder')}_{Path(row.get('bg_path', '')).stem}"
        lines.append(f"| {scenario_id} | {row.get('path', '')} | {row.get('bg_path', '')} |")
    audit_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("official_scenarios =", len(official_rows))
    print("audit_path =", audit_path)
else:
    print("RUN_AUDIT is False; audit skipped.")

## 9. Create Official All-Demos Manifest

Writes the formal 20-scenario ladder manifest and first-scenario smoke manifest under `EXP_ROOT`.

In [ ]:
manifest_path = EXP_ROOT / "manifests" / "unilumos_official_all_demos_manifest.csv"
manifest_jsonl_path = EXP_ROOT / "manifests" / "unilumos_official_all_demos_manifest.jsonl"
smoke_manifest_path = EXP_ROOT / "manifests" / "unilumos_official_smoke_manifest.csv"

need_manifest = not manifest_path.exists() or not smoke_manifest_path.exists()
if RUN_MAKE_MANIFEST and (RESET_MANIFEST or need_manifest):
    manifest_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "make_manifest_unilumos_official_all_demos.py"),
        "--experiment_root", str(EXP_ROOT),
        "--mode", "abc",
    ]
    manifest_output = subprocess.check_output(manifest_cmd, text=True).strip().splitlines()
    print("\n".join(manifest_output))
elif RUN_MAKE_MANIFEST:
    print("Existing manifests found; keeping them for resume. Set RESET_MANIFEST=True to recreate from scratch.")
else:
    print("RUN_MAKE_MANIFEST is False; using existing manifests.")

print("manifest_path =", manifest_path)
print("smoke_manifest_path =", smoke_manifest_path)
assert manifest_path.exists(), manifest_path
assert smoke_manifest_path.exists(), smoke_manifest_path

## 10. Optional Smoke Dry-Run

Skipped by default because `RUN_SMOKE_FIRST = False`. Enable only if you want a first-scenario smoke gate before the full run.

In [ ]:
if RUN_SMOKE_FIRST:
    dry_run_smoke_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "run_manifest.py"),
        "--manifest", str(smoke_manifest_path),
        "--weights_root", str(WEIGHTS_ROOT),
        "--dry_run",
    ]
    subprocess.check_call(dry_run_smoke_cmd)
    print("Dry-run report:", EXP_ROOT / "reports" / "01_dry_run_commands.md")
else:
    print("RUN_SMOKE_FIRST is False; smoke dry-run skipped.")

## 11. Optional Smoke Gate

Skipped by default. When `RUN_SMOKE_FIRST = False`, the full manifest runs all rows directly.

In [ ]:
def read_manifest_rows(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def manifest_status(path: Path) -> Counter:
    return Counter(row.get("status", "") for row in read_manifest_rows(path))

if RUN_SMOKE_FIRST:
    smoke_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "run_manifest.py"),
        "--manifest", str(smoke_manifest_path),
        "--weights_root", str(WEIGHTS_ROOT),
        "--resume",
    ]
    subprocess.check_call(smoke_cmd)
    print("smoke_status =", dict(manifest_status(smoke_manifest_path)))
else:
    print("RUN_SMOKE_FIRST is False; smoke execution skipped. Full manifest will run all rows directly.")
    if smoke_manifest_path.exists():
        print("smoke_manifest_status =", dict(manifest_status(smoke_manifest_path)))

## 12. Optional Smoke Validation And Merge

Skipped by default. When smoke is disabled, the full manifest remains responsible for all rows, including the first scenario.

In [ ]:
def write_manifest_rows(path: Path, rows: list[dict]) -> None:
    fields = []
    for row in rows:
        for key in row:
            if key not in fields:
                fields.append(key)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

if RUN_SMOKE_FIRST:
    smoke_rows = read_manifest_rows(smoke_manifest_path)
    failed_smoke = [row for row in smoke_rows if row.get("status") != "done"]
    if failed_smoke:
        for row in failed_smoke:
            print(row.get("run_id"), row.get("status"), row.get("error_message"))
        raise RuntimeError("Smoke gate did not complete; do not run full manifest yet.")

    smoke_schedule_paths = [row["schedule_json_path"] for row in smoke_rows]
    validate_smoke_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "validate_schedules.py"),
        *smoke_schedule_paths,
    ]
    subprocess.check_call(validate_smoke_cmd)

    full_rows = read_manifest_rows(manifest_path)
    smoke_by_run_id = {row["run_id"]: row for row in smoke_rows}
    merged = 0
    for row in full_rows:
        smoke_row = smoke_by_run_id.get(row.get("run_id"))
        if not smoke_row:
            continue
        for key in ("status", "error_message"):
            row[key] = smoke_row.get(key, row.get(key, ""))
        merged += 1
    write_manifest_rows(manifest_path, full_rows)
    print("merged_smoke_rows_into_full_manifest =", merged)
    print("full_manifest_status =", dict(manifest_status(manifest_path)))

    report_path = EXP_ROOT / "reports" / "01_smoke_report.md"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(
        "# UniLumos Official All-Demos Smoke Report\n\n"
        f"- smoke_manifest: `{smoke_manifest_path}`\n"
        f"- rows: {len(smoke_rows)}\n"
        f"- status: all done\n"
        f"- schedule_json_count: {len(smoke_schedule_paths)}\n"
        f"- full_manifest_merged_rows: {merged}\n",
        encoding="utf-8",
    )
    print("smoke_report =", report_path)
else:
    full_rows = read_manifest_rows(manifest_path)
    print("RUN_SMOKE_FIRST is False; smoke validation/merge skipped.")
    print("full_manifest_status =", dict(manifest_status(manifest_path)))
    print("Full run will execute pending rows from the full manifest directly.")

## 13. Dry-Run / Inspect Full Manifest

Full manifest is 20 official scenarios x 12 methods = 240 planned rows. Use `FULL_LIMIT` for a short trial if needed.

In [ ]:
full_rows = read_manifest_rows(manifest_path)
print("total_rows =", len(full_rows))
print("status =", dict(manifest_status(manifest_path)))
print("methods =", dict(Counter(row.get("method") for row in full_rows)))
print("scenarios =", len({row.get("scenario_id") or row.get("case_id") for row in full_rows}))

full_dry_run_cmd = [
    sys.executable,
    str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "run_manifest.py"),
    "--manifest", str(manifest_path),
    "--weights_root", str(WEIGHTS_ROOT),
    "--dry_run",
]
if FULL_LIMIT is not None:
    full_dry_run_cmd.extend(["--limit", str(FULL_LIMIT)])
if ONLY_METHODS:
    full_dry_run_cmd.extend(["--only_methods", ONLY_METHODS])
subprocess.check_call(full_dry_run_cmd)

## 14. Run Full Official-Demo Ladder

This can take a long time. It resumes completed rows and writes runtime/status after every row.

In [ ]:
if RUN_FULL:
    full_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "run_manifest.py"),
        "--manifest", str(manifest_path),
        "--weights_root", str(WEIGHTS_ROOT),
        "--resume",
    ]
    if FULL_LIMIT is not None:
        full_cmd.extend(["--limit", str(FULL_LIMIT)])
    if ONLY_METHODS:
        full_cmd.extend(["--only_methods", ONLY_METHODS])
    subprocess.check_call(full_cmd)
else:
    print("RUN_FULL is False; full run skipped.")

print("full_manifest_status =", dict(manifest_status(manifest_path)))

## 15. Validate Full Schedules And Write Run Report

Validates completed schedule JSON files and writes `reports/02_full_official_demos_run_report.md`.

In [ ]:
def runtime_seconds(row: dict):
    path = Path(row.get("runtime_json_path", ""))
    if not path.exists():
        return None
    try:
        return float(json.loads(path.read_text(encoding="utf-8")).get("runtime_seconds"))
    except Exception:
        return None

full_rows = read_manifest_rows(manifest_path)
done_rows = [row for row in full_rows if row.get("status") == "done"]
failed_rows = [row for row in full_rows if row.get("status") == "failed"]
schedule_paths = [row["schedule_json_path"] for row in done_rows if Path(row["schedule_json_path"]).exists()]
print("completed_rows =", len(done_rows), "failed_rows =", len(failed_rows), "schedule_json_count =", len(schedule_paths))
if schedule_paths:
    validate_full_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "validate_schedules.py"),
        *schedule_paths,
    ]
    subprocess.check_call(validate_full_cmd)

per_method_runtime = defaultdict(list)
for row in done_rows:
    sec = runtime_seconds(row)
    if sec is not None:
        per_method_runtime[row.get("method")].append(sec)

report_lines = [
    "# UniLumos Official Demos Full Run Report",
    "",
    f"- selected mode: `abc`",
    f"- official scenarios: {len({row.get('scenario_id') or row.get('case_id') for row in full_rows})}",
    f"- total rows planned: {len(full_rows)}",
    f"- completed rows: {len(done_rows)}",
    f"- failed rows: {len(failed_rows)}",
    f"- schedule JSONs: {len(schedule_paths)}",
    f"- total runtime seconds: {sum(sum(values) for values in per_method_runtime.values()):.1f}",
    "",
    "## Per-method runtime",
    "",
    "| method | completed | mean_runtime_sec |",
    "|---|---:|---:|",
]
for method in sorted(per_method_runtime):
    values = per_method_runtime[method]
    report_lines.append(f"| {method} | {len(values)} | {sum(values)/len(values):.1f} |")
if failed_rows:
    report_lines.extend(["", "## Failures", "", "| run_id | error |", "|---|---|"])
    for row in failed_rows:
        report_lines.append(f"| {row.get('run_id')} | {row.get('error_message')} |")
report_path = EXP_ROOT / "reports" / "02_full_official_demos_run_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("full_run_report =", report_path)

## 16. Compute Metrics

Reference is `reference_uniform25`; low baseline is `uniform8`. Writes long and per-case metrics.

In [ ]:
if RUN_METRICS:
    metrics_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "compute_metrics_against_ref.py"),
        "--manifest", str(manifest_path),
    ]
    metrics_output = subprocess.check_output(metrics_cmd, text=True).strip().splitlines()
    print("\n".join(metrics_output))
    metrics_csv = Path(metrics_output[0])
    per_case_metrics_csv = Path(metrics_output[1]) if len(metrics_output) > 1 else EXP_ROOT / "metrics" / "per_case_metrics.csv"
    master_long_metrics_csv = Path(metrics_output[2]) if len(metrics_output) > 2 else EXP_ROOT / "metrics" / "master_long_metrics.csv"
else:
    metrics_csv = EXP_ROOT / "metrics" / "metrics_against_ref.csv"
    per_case_metrics_csv = EXP_ROOT / "metrics" / "per_case_metrics.csv"
    master_long_metrics_csv = EXP_ROOT / "metrics" / "master_long_metrics.csv"
    print("RUN_METRICS is False; using existing metrics paths.")

print("metrics_csv =", metrics_csv, metrics_csv.exists())
print("per_case_metrics_csv =", per_case_metrics_csv, per_case_metrics_csv.exists())
print("master_long_metrics_csv =", master_long_metrics_csv, master_long_metrics_csv.exists())

## 17. Generate Paper Tables

Generates Table 1A, Table 1B, Table 2, detailed CSVs, and the cross-model summary row.

In [ ]:
if RUN_TABLES:
    table_cmd = [
        sys.executable,
        str(TASK_ROOT / "bss_experiments" / "unilumos" / "scripts" / "make_cross_model_tables.py"),
        "--metrics_csv", str(metrics_csv),
    ]
    table_output = subprocess.check_output(table_cmd, text=True).strip().splitlines()
    print("\n".join(table_output))
else:
    table_output = []
    print("RUN_TABLES is False; table generation skipped.")

table_paths = {}
for line in table_output:
    if "=" in line:
        key, value = line.split("=", 1)
        table_paths[key.strip()] = Path(value.strip())

# Stable defaults, useful when re-running this summary cell later.
table1a_path = table_paths.get("table1a_md", EXP_ROOT / "tables" / "table1a_same_compute_rgb_closure_main.md")
table1b_path = table_paths.get("table1b_md", EXP_ROOT / "tables" / "table1b_same_compute_rgb_closure_support_aware.md")
table2_path = table_paths.get("table2_md", EXP_ROOT / "tables" / "table2_matched_quality_compute_saving.md")
summary_row_path = table_paths.get("summary_md", EXP_ROOT / "tables" / "cross_model_summary_row.md")
print("table1a_path =", table1a_path, table1a_path.exists())
print("table1b_path =", table1b_path, table1b_path.exists())
print("table2_path =", table2_path, table2_path.exists())
print("summary_row_path =", summary_row_path, summary_row_path.exists())

## 18. Print Final Summary And Table Preview

Use this cell after full metrics/tables finish. It prints the paths needed for the paper/report handoff.

In [ ]:
def show_markdown_file(path: Path, title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print(path)
    print("=" * 80)
    if not path.exists():
        print("missing")
        return
    text = path.read_text(encoding="utf-8")
    try:
        from IPython.display import Markdown, display
        display(Markdown(f"**{title}**\n\n" + text))
    except Exception:
        print(text)

if RUN_FINAL_SUMMARY:
    rows = read_manifest_rows(manifest_path)
    status = Counter(row.get("status") for row in rows)
    scenarios = {row.get("scenario_id") or row.get("case_id") for row in rows}
    schedules = [row for row in rows if Path(row.get("schedule_json_path", "")).exists()]
    print("manifest path:", manifest_path)
    print("metrics path:", metrics_csv)
    print("table1A path:", table1a_path)
    print("table1B path:", table1b_path)
    print("table2 path:", table2_path)
    print("cross_model_summary_row path:", summary_row_path)
    print("selected mode: abc")
    print("official scenarios:", len(scenarios))
    print("row status:", dict(status))
    print("schedule json count:", len(schedules))
    print("full run report:", EXP_ROOT / "reports" / "02_full_official_demos_run_report.md")
    show_markdown_file(table1a_path, "Table 1A Same-Compute RGB Closure Main")
    show_markdown_file(table1b_path, "Table 1B Strict Support-Aware RGB Closure")
    show_markdown_file(table2_path, "Table 2 Matched-Quality Compute Saving")
    show_markdown_file(summary_row_path, "Cross-Model Summary Row")
else:
    print("RUN_FINAL_SUMMARY is False; final summary skipped.")